## **DS Graph notebook** 

### Imports

In [1]:
#imports

import csv

import math
import numpy as np
import pandas as pd
import psycopg2
import neo4j
import json
import ast
from IPython.display import display

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

### Set up postgres

In [2]:
#postgres
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)
cursor = connection.cursor()

In [3]:
def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)

In [4]:
def delimit_single_quotes(string_value):
    return string_value.replace("'", "\\'")

In [5]:
def words_from_list(list_of_words):
    if list_of_words == "Unknown":
        return list_of_words
    text_to_return = ""
    for count, word in enumerate(ast.literal_eval(list_of_words)):
        text_to_return += word
        if count < len(ast.literal_eval(list_of_words)) - 1:
            text_to_return += ", "
    return text_to_return

In [6]:
def format_text(title, budget, genres, unused_id, keywords, overview, popularity, vote_average, vote_count, 
                cast, director, executive_producer, writer, original_music_composer, director_photography, 
                production_country, production_company, languages):
    return f"{title} (Genres: {genres}. Keywords: {keywords}): {overview} {title} is popular among {popularity} percent of viewers. {vote_count} viewers gave this movie an average rating of {vote_average} out of 10. The cast includes {cast}. {title} was directed by {director} with {words_from_list(executive_producer)} as the executive producer(s). The movie was written by {words_from_list(writer)}. The music was originally composed by {words_from_list(original_music_composer)}, and the photography was directed by {words_from_list(director_photography)}. {title} was produced with a budget of {budget} by {words_from_list(production_company)} in {words_from_list(production_country)}."

In [7]:
def get_embeddings(text):
    model = SentenceTransformer("all-MiniLM-L6-v2") # Source: https://sbert.net/docs/sentence_transformer/usage/usage.html
    embeddings = model.encode(text) # Result in a 1D array with 384 elements.
    embeddings_array = np.array(embeddings)
    print("Shape of embeddings:", embeddings_array.shape)
    return embeddings_array

### Set up clean movies table

In [8]:
connection.rollback()

query = """

drop table if exists cleanmovies

"""

cursor.execute(query)

connection.commit()

In [9]:
connection.rollback()

query = """

create table cleanmovies (
    index numeric,
    title text,
    budget numeric,
    genres text,
    id numeric,
    keywords text,
    overview text,
    popularity numeric,
    vote_average numeric,
    vote_count numeric,
    cast_ text,
    director text,
    executive_producer text,
    writer text,
    original_music_composer text,
    director_photography text,
    production_country text,
    production_company varchar,
    languages text  
)

"""

cursor.execute(query)

connection.commit()

In [10]:
connection.rollback()

query = """

copy cleanmovies
from '/user/projects/project-3-richardshelby/data/interim/cleanmovies.csv' delimiter ',' NULL '' csv header;

"""

cursor.execute(query)

connection.commit()

In [11]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select *
from cleanmovies

"""
cleanmovies = my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

In [12]:
for i in cleanmovies:
    print(i)

index
title
budget
genres
id
keywords
overview
popularity
vote_average
vote_count
cast_
director
executive_producer
writer
original_music_composer
director_photography
production_country
production_company
languages


In [13]:
vector_embeddings = pd.read_csv("../data/interim/Summarized_Text_3.0_Vector_Embeddings.csv")

In [14]:
vector_embeddings_clean = np.nan_to_num(vector_embeddings.values)
similarities = cosine_similarity(vector_embeddings_clean, vector_embeddings_clean)
pd.DataFrame(similarities)

,0,1,2,3,4,5,6,7,8,9,...,3213,3214,3215,3216,3217,3218,3219,3220,3221,3222
0,1.000000,0.212203,0.120765,0.094107,0.078594,0.058424,0.040448,0.043880,0.026294,0.024612,...,0.000063,0.000104,0.000084,0.000065,0.000034,0.000085,0.000132,0.000126,0.000111,0.000106
1,0.212203,1.000000,0.707875,0.733565,0.720402,0.718863,0.736824,0.710632,0.720956,0.723620,...,0.707149,0.707156,0.707161,0.707159,0.707129,0.707175,0.707151,0.707145,0.707154,0.707168
2,0.120765,0.707875,1.000000,0.867063,0.889453,0.901552,0.897012,0.894326,0.904186,0.895127,...,0.894467,0.894461,0.894463,0.894473,0.894456,0.894451,0.894457,0.894454,0.894459,0.894446
3,0.094107,0.733565,0.867063,1.000000,0.950010,0.946570,0.950215,0.950895,0.949553,0.955450,...,0.948703,0.948706,0.948730,0.948711,0.948705,0.948712,0.948702,0.948704,0.948703,0.948697
4,0.078594,0.720402,0.889453,0.950010,1.000000,0.965180,0.968973,0.972014,0.971197,0.977193,...,0.970165,0.970167,0.970172,0.970165,0.970156,0.970161,0.970159,0.970157,0.970162,0.970163
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3218,0.000085,0.707175,0.894451,0.948712,0.970161,0.980600,0.986409,0.989959,0.992288,0.993893,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3219,0.000132,0.707151,0.894457,0.948702,0.970159,0.980596,0.986406,0.989962,0.992286,0.993889,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3220,0.000126,0.707145,0.894454,0.948704,0.970157,0.980594,0.986407,0.989960,0.992287,0.993889,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
3221,0.000111,0.707154,0.894459,0.948703,0.970162,0.980597,0.986405,0.989960,0.992286,0.993892,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [15]:
cleanmovies = cleanmovies.fillna('Unknown')

In [16]:
cleanmovies['title'][123]

'American Psycho'

### Neo4j set up (using code from 1.0-rjs-eda and 3.0-ah-embeddings-eda)

In [17]:
driver = neo4j.GraphDatabase.driver(uri="neo4j://neo4j:7687", auth=("neo4j","ucb_mids_w205"))

In [18]:
session = driver.session(database="neo4j")

In [19]:
def my_neo4j_wipe_out_database():
    "wipe out database by deleting all nodes and relationships"
    
    query = "match (node)-[relationship]->() delete node, relationship"
    session.run(query)
    
    query = "match (node) delete node"
    session.run(query)

In [20]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [21]:
def my_neo4j_nodes_relationships():
    "print all the nodes and relationships"
   
    print("-------------------------")
    print("  Nodes:")
    print("-------------------------")
    
    query = """
        match (n) 
        return n.title as movie_title, labels(n) as labels
        order by n.title
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    display(df)
    
    print("-------------------------")
    print("  Relationships:")
    print("-------------------------")
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.title as movie_title_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.title as movie_title_2, labels(n2) as node_2_labels
        order by movie_title_1, movie_title_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    display(df)
    
    density = (2 * number_relationships) / (number_nodes * (number_nodes - 1))
    
    print("-------------------------")
    print("  Density:", f'{density:.1f}')
    print("-------------------------")

In [22]:
def my_neo4j_create_relationship_two_way(from_movie_id, to_movie_id, weight):
    "create relationships two way between two movies with a weight"
    
    query = """
    
    MATCH (from:Movie), 
          (to:Movie)
    WHERE from.id = $from_movie_id and to.id = $to_movie_id
    MERGE (from)-[r1:SIMILAR]->(to)
    MERGE (to)-[r2:SIMILAR]->(from)
    SET r1.weight = $weight,
        r2.weight = $weight
    
    """
    
    session.run(query, from_movie_id=from_movie_id, to_movie_id=to_movie_id, weight=weight)

In [23]:
def my_neo4j_create_movie_nodes(movies, movie_count):
    """
        movies: pandas dataframe of movies
        movie_count: the number of movies to create nodes for
    """

    nquery = "CREATE \n"
    first = True

    for _, movie in movies.head(movie_count).iterrows():
        if first:
            first = False
        else:
            nquery += ",\n"
            nquery += "(:Movie {"
            nquery += f"title: '{delimit_single_quotes(movie['title'])}', "
            nquery += f"id: {movie['id']}, "
            nquery += f"budget: {movie['budget']}, "
            # this isn't completely accurate. the kaggle set shows new lines but the CSV does not 
            #    have this information. so we have no way to differentiate a movie listed with a 
            #    single genre like "Action Adventure" from one with two genres "Action" and "Adventure"
            #    probably fine, but may be worth mentioning
            nquery += f"genres: '{delimit_single_quotes(movie['genres'])}', "
            nquery += f"keywords: '{delimit_single_quotes(movie['keywords'])}', "
            nquery += f"overview: '{delimit_single_quotes(movie['overview'])}', "
            nquery += f"popularity: {movie['popularity']}, "
            nquery += f"vote_average: {movie['vote_average']}, "
            nquery += f"vote_count: {movie['vote_count']}, "

            # same problem, no delimiter between people. 
            #    space used to seperate first name, last name, and people
            nquery += f"movie_cast: '{delimit_single_quotes(movie['cast_'])}', "

            nquery += f"director: '{delimit_single_quotes(movie['director'])}', "
            nquery += f"director: '{delimit_single_quotes(movie['director'])}', "
            nquery += f"production_company: '{delimit_single_quotes(movie['production_company'])}', "
            
            nquery += "})"

    nquery += ";"
    session.run(nquery)

In [24]:
def my_neo4j_create_movie_nodes(movies, movie_count): 
    """
        movies: pandas dataframe of movies
        movie_count: the number of movies to create nodes for
    """

    nquery = "CREATE \n"
    first = True

    for _, movie in movies.head(movie_count).iterrows():
        if first:
            first = False
        else:
            nquery += ",\n"
        nquery += "(:Movie {"
        nquery += f"title: '{delimit_single_quotes(movie['title'])}', "
        nquery += f"budget: {movie['budget']}, "
        nquery += f"genres: '{delimit_single_quotes(movie['genres'])}', "
        nquery += f"id: {movie['id']}, "
        nquery += f"keywords: '{delimit_single_quotes(movie['keywords'])}', "
        nquery += f"overview: '{delimit_single_quotes(movie['overview'])}', "
        nquery += f"popularity: {movie['popularity']}, "
        nquery += f"vote_average: {movie['vote_average']}, "
        nquery += f"vote_count: {movie['vote_count']}, "
        nquery += f"movie_cast: '{delimit_single_quotes(movie['cast_'])}', "
        nquery += f"director: '{delimit_single_quotes(movie['director'])}', "
        nquery += f"executive_producer: '{delimit_single_quotes(words_from_list(movie['executive_producer']))}', "
        nquery += f"writer: '{delimit_single_quotes(words_from_list(movie['writer']))}', "
        nquery += f"original_music_composer: '{delimit_single_quotes(words_from_list(movie['original_music_composer']))}', "
        nquery += f"director_photography: '{delimit_single_quotes(words_from_list(movie['director_photography']))}', "
        nquery += f"production_country: '{delimit_single_quotes(words_from_list(movie['production_country']))}', "
        nquery += f"production_company: '{delimit_single_quotes(words_from_list(movie['production_company']))}'"
        nquery += "})"
    nquery += ";"
    session.run(nquery)

In [25]:
def my_neo4j_create_top_3_similarity_links(movies, movie_count, cosine_similarity_matrix):
    """
        movies: pandas dataframe of movies
        movie_count: the number of movies to create nodes for        
        cosine_similarity_matrix: similarity matrix in the range [-1, 1] where 
            -1 is opposite taste
             0 is no similarity
             1 is perfect match
             
        note - this is hardcoded to top 3 but we can make it general if needed
    """

    for movie_idx in range(movie_count):
        # track top 3 movie by index and score
        similar_1_idx = -1
        similar_2_idx = -1
        similar_3_idx = -1
        similar_1_score = -1
        similar_2_score = -1
        similar_3_score = -1

        for movie_idx_compare in range(movie_count):
            
            # only compare movies to other movies
            if movie_idx == movie_idx_compare:
                continue
                
            similarity_score = cosine_similarity_matrix[movie_idx][movie_idx_compare]

            if similarity_score > similar_1_score:
                similar_3_idx = similar_2_idx
                similar_3_score = similar_2_score

                similar_2_idx = similar_1_idx            
                similar_2_score = similar_1_score

                similar_1_idx = movie_idx_compare            
                similar_1_score = similarity_score

            elif similarity_score > similar_2_score:
                similar_3_idx = similar_2_idx
                similar_3_score = similar_2_score

                similar_2_idx = movie_idx_compare
                similar_2_score = similarity_score

            elif similarity_score > similar_3_score:
                similar_3_idx = movie_idx_compare
                similar_3_score = similarity_score

        most_similar_array = [
            (similar_1_idx, similar_1_score),
            (similar_2_idx, similar_2_score),
            (similar_3_idx, similar_3_score)
        ]
        
        for similar_idx, similar_score in most_similar_array:
            my_neo4j_create_relationship_two_way(
                from_movie_id = movies.iloc[movie_idx].id, 
                to_movie_id = movies.iloc[similar_idx].id, 
                weight = round(similar_score, 5)
            )

### Set up graph 

In [26]:
my_neo4j_wipe_out_database()

In [27]:
my_neo4j_create_movie_nodes(cleanmovies,len(cleanmovies))

In [28]:
my_neo4j_create_top_3_similarity_links(cleanmovies, len(cleanmovies), similarities)

In [29]:
my_neo4j_nodes_relationships()

-------------------------
  Nodes:
-------------------------


,movie_title,labels
0,(500) Days of Summer,[Movie]
1,10 Cloverfield Lane,[Movie]
2,10 Things I Hate About You,[Movie]
3,102 Dalmatians,[Movie]
4,10th & Wolf,[Movie]
...,...,...
3218,[REC]²,[Movie]
3219,eXistenZ,[Movie]
3220,xXx,[Movie]
3221,xXx: State of the Union,[Movie]


-------------------------
  Relationships:
-------------------------


,movie_title_1,node_1_labels,relationship_type,movie_title_2,node_2_labels
0,(500) Days of Summer,[Movie],SIMILAR,Agora,[Movie]
1,(500) Days of Summer,[Movie],SIMILAR,Alatriste,[Movie]
2,(500) Days of Summer,[Movie],SIMILAR,Amélie,[Movie]
3,10 Cloverfield Lane,[Movie],SIMILAR,102 Dalmatians,[Movie]
4,10 Cloverfield Lane,[Movie],SIMILAR,10th & Wolf,[Movie]
...,...,...,...,...,...
17467,xXx: State of the Union,[Movie],SIMILAR,"You, Me and Dupree",[Movie]
17468,Æon Flux,[Movie],SIMILAR,3000 Miles to Graceland,[Movie]
17469,Æon Flux,[Movie],SIMILAR,A Perfect Getaway,[Movie]
17470,Æon Flux,[Movie],SIMILAR,Argo,[Movie]


-------------------------
  Density: 0.0
-------------------------


# Page Rank code 

In [30]:
query = "CALL gds.graph.drop('ds_graph', false) yield graphName"
session.run(query)

In [31]:
query = "CALL gds.graph.project('ds_graph', 'Movie', 'SIMILAR', {relationshipProperties: 'weight'})"
session.run(query)

In [32]:
query = """

CALL gds.pageRank.stream('ds_graph',
                         { maxIterations: $max_iterations,
                           dampingFactor: $damping_factor
                           }
                         )
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).title AS title, 
       gds.util.asNode(nodeId).popularity AS movie_popularity, 
       gds.util.asNode(nodeId).vote_average AS movie_score, 
       score as page_rank
ORDER BY page_rank DESC, movie_popularity ASC, movie_score DESC

"""

max_iterations = 20
damping_factor = 0.5

influence_rank = my_neo4j_run_query_pandas(query, max_iterations=max_iterations, damping_factor=damping_factor)
influence_rank.head(25)

,title,movie_popularity,movie_score,page_rank
0,Vampires Suck,17.165039,4.2,5.723216
1,Warrior,51.915025,7.7,5.256568
2,Zombieland,57.300674,7.2,4.461851
3,Wanted,73.822890,6.4,4.390922
4,You Don't Mess with the Zohan,40.597344,5.5,4.257504
5,Zero Dark Thirty,38.306954,6.7,4.249646
6,Waterloo,1.894749,7.0,4.005766
7,Waltz with Bashir,14.082510,7.8,4.002717
8,Vicky Cristina Barcelona,32.758254,6.7,3.394061
9,What Happens in Vegas,38.100488,5.8,3.272805


In [34]:
query = """
MATCH (Movie:Movie {title: $source})
CALL gds.pageRank.stream('ds_graph',
                         { maxIterations: $max_iterations,
                           dampingFactor: $damping_factor,
                           sourceNodes: [Movie]
                           }
                         )
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).title AS title, 
       gds.util.asNode(nodeId).popularity AS movie_popularity, 
       gds.util.asNode(nodeId).vote_average AS movie_score, 
       score as page_rank
ORDER BY page_rank DESC, movie_popularity ASC, movie_score DESC

"""

source = cleanmovies['title'][np.random.randint(0,3224)]
#source = 'Vampires Suck'
max_iterations = 20
damping_factor = 0.5

print(source)

personalized_influence_rank = my_neo4j_run_query_pandas(query, max_iterations=max_iterations, damping_factor=damping_factor, source=source)
personalized_influence_rank.head(25)

The Best Little Whorehouse in Texas


,title,movie_popularity,movie_score,page_rank
0,The Best Little Whorehouse in Texas,4.189378,5.9,0.520035
1,The Tree of Life,43.862456,6.5,0.092327
2,The Secret Life of Pets,31.482872,5.9,0.091222
3,Pi,27.788067,7.1,0.088823
4,Titan A.E.,14.443810,6.3,0.009560
5,W.,10.445391,6.1,0.009452
6,Transcendence,58.991388,5.9,0.009406
7,The Thing,52.731379,7.8,0.009338
8,Close Encounters of the Third Kind,52.456505,7.2,0.009312
9,The Secret Life of Bees,7.645979,7.4,0.009033


In [ ]:
source = cleanmovies['title'][np.random.randint(0,3224)]

source

In [ ]:
type(source)